# Experiment A — mimic-code UO labels (non-destructive)
Tests whether adopting mimic-code's `uo_tm` urine method (instead of our strict
rolling-window UO arm) changes the headline result. **Reads** `aki_cohort_v2.parquet`,
**writes** a separate `aki_cohort_v2_mcuo.parquet` — your main notebook and parquet
are untouched, and `..._baseline` copies are preserved.

**How to run:** Run All. Cell 2 relabels the UO arm on a sample
(`EXP_STAYS=10000` by default; set `EXP_STAYS=0` for the full cohort) — it re-streams
`outputevents` only (the small table), budget ~10-20 min in a no-DuckDB kernel.
Cell 3 prints the side-by-side AUROC/AUPRC and writes `uo_experiment_comparison.csv`.

**Reading it:** if mimic-code-UO AUROC is close to current → the more-defensible
labels cost little, adopt them. If it drops materially → keep current labels and
report the UO divergence as a limitation.


## 1 · Engine setup (same code as the main notebook)


In [1]:
# ============================================================================
# 1 · CONFIG  —  MIMIC item IDs + v3.3-faithful engine knobs (KDIGO) + fixes
# ============================================================================
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd

# --- Paths (MIMIC auto-detect) ----------------------------------------------
MIMIC_CANDIDATES = [Path("/Users/sumiyasayeed/Documents/mimic4/mimiciv"),
                    Path("/Users/sumiyasayeed/Documents/mimic4")]
MIMIC_ROOT = next((p for p in MIMIC_CANDIDATES if p.exists()), MIMIC_CANDIDATES[-1])
HOSP_DIR, ICU_DIR = MIMIC_ROOT / "hosp", MIMIC_ROOT / "icu"
PROJECT_ROOT = Path.cwd()
DERIVED_DIR = PROJECT_ROOT / "derived"; DERIVED_DIR.mkdir(exist_ok=True)
# v2 = after the label/ETL fixes. The old aki_cohort.parquet (buggy labels) is
# left untouched and is NOT loaded, so stale results can't sneak back in.
COHORT_PARQUET = DERIVED_DIR / "aki_cohort_v2.parquet"

# --- MIMIC-IV item IDs ------------------------------------------------------
CREATININE_ITEMIDS = [50912, 52546]                 # hosp/labevents, mg/dL
BUN_ITEMIDS        = [51006]
# Urine output — canonical MIT-LCP pivoted-UO set. GU Irrigant *In* (227488) is
# fluid going IN and must be SUBTRACTED, not counted as output (FIX).
URINE_POSITIVE_ITEMIDS = [226559, 226560, 226561, 226584, 226563, 226564,
                          226565, 226567, 226557, 226558, 227489]
URINE_NEGATIVE_ITEMIDS = [227488]                   # GU Irrigant Volume In -> subtract
UO_ITEMIDS = URINE_POSITIVE_ITEMIDS + URINE_NEGATIVE_ITEMIDS
WEIGHT_ITEMIDS = [226512, 224639]                   # Admit Wt, Daily Wt (kg)
MAP_ITEMIDS    = [220052, 220181, 225312]
VASO_ITEMIDS   = [221906, 221749, 221662, 222315, 221289]
NEPHROTOX_IV_ITEMIDS = [225798, 225875, 225902, 221794]   # vanco/genta/tobra/furosemide
RRT_PROCEDURE_ITEMIDS = [225441, 225802, 225803, 225805, 225809, 225955]  # HD/CRRT/CVVHD/PD/CVVHDF/SCUF

# Physiologic validity filter — drop implausible raw values before staging/emission.
WEIGHT_VALID = (20.0, 300.0)        # kg
UO_RATE_CLIP = (0.0, 20.0)          # mL/kg/h (anything above is a charting artifact)
# Urine output is the strongest predictor but charted only ~52% of hours. A
# causal (trailing) rolling mean of this many hours lifts emission coverage to
# ~86% and denoises it (validated: +~0.05 AUROC, +~0.04 AUPRC). Set 0 to disable.
UO_EMIT_SMOOTH_H = 6

# --- Multi-state HMM: the 4 ordered KDIGO regimes ---------------------------
STATE_LABELS = ["No_AKI", "KDIGO_1", "KDIGO_2", "KDIGO_3"]
N_STATES = len(STATE_LABELS)
STATE_PROGRESSION = np.array([0.0, 1.0, 2.0, 3.0])   # z-units (used only if EMISSION_MODE='zscore')
ABSORBING_RRT_STATE = 3

FEATURES = ["creatinine_ratio", "dSCr_48h", "uo_rate", "bun"]
FEATURE_DIRECTION = {"creatinine_ratio": +1, "dSCr_48h": +1, "uo_rate": -1, "bun": +1}
FEATURE_WEIGHTS   = {"creatinine_ratio": 1.6, "dSCr_48h": 1.3, "uo_rate": 1.5, "bun": 1.0}
VALID_RANGES = {"creatinine_ratio": (0.2, 15), "dSCr_48h": (-2, 15),
                "uo_rate": (0.0, 20), "bun": (1, 250)}

# --- Emission model ----------------------------------------------------------
# 'clinical' (default, FIX): per-state Gaussian means anchored to KDIGO clinical
#   thresholds, in RAW units — strong, interpretable signal.
# 'zscore'  : the original v3.3 geometry (z-score vs population × STATE_PROGRESSION
#   × FEATURE_DIRECTION). Kept for faithful comparison; flattened the signal on
#   real AKI data because the population already contains heavy AKI tails.
EMISSION_MODE = "clinical"
# (means, sds) per state for each feature. creatinine_ratio means track KDIGO
# ratio bands (1.5-1.9 / 2.0-2.9 / >=3); uo_rate tracks the <0.5 / <0.3 cut-offs.
CLINICAL_ANCHORS = {
    "creatinine_ratio": ([1.00, 1.60, 2.40, 3.50], [0.30, 0.35, 0.50, 0.90]),
    "dSCr_48h":         ([0.00, 0.35, 0.60, 1.00], [0.15, 0.20, 0.30, 0.50]),
    "uo_rate":          ([1.00, 0.45, 0.30, 0.10], [0.40, 0.18, 0.13, 0.08]),
    "bun":              ([15.0, 30.0, 45.0, 70.0], [9.0, 12.0, 16.0, 22.0]),
}
# Population z-score fallback (only used in 'zscore' mode); filled from TRAIN.
GLOBAL_STATS = {"creatinine_ratio": (1.0, 0.5), "dSCr_48h": (0.1, 0.4),
                "uo_rate": (0.9, 0.5), "bun": (25.0, 18.0)}

AGE_BRACKETS = [(0, 45), (45, 65), (65, 80), (80, 200)]
PRIOR_BELIEF = np.array([0.85, 0.10, 0.04, 0.01])

# --- KDIGO 2012 thresholds ---------------------------------------------------
@dataclass(frozen=True)
class KDIGOThresholds:
    abs_rise_48h: float = 0.3; ratio_stage1: float = 1.5
    ratio_stage2: float = 2.0; ratio_stage3: float = 3.0; scr_stage3_abs: float = 4.0
    uo_low_rate: float = 0.5; uo_s1_hours: int = 6; uo_s2_hours: int = 12
    uo_s3_rate: float = 0.3; uo_s3_hours: int = 24; anuria_hours: int = 12
KDIGO = KDIGOThresholds()

@dataclass
class CohortConfig:
    min_age: int = 18; min_los_days: float = 1.5; min_creatinine_measurements: int = 2
    default_egfr: float = 75.0; grid_freq: str = "1h"; nephrotox_halflife_h: float = 24.0
COHORT = CohortConfig()

@dataclass
class EvalConfig:
    horizons_h: tuple = (12, 24, 48)
    # 200 resamples give stable 95% CIs at this scale (~3 min); raise BOOTSTRAP_N
    # for tighter intervals, lower it to go faster.
    n_bootstrap: int = int(os.environ.get("BOOTSTRAP_N", "200"))
    bootstrap_alpha: float = 0.05; random_state: int = 7
EVAL = EvalConfig()

# --- Layer 2 (hierarchical personalized emissions) ---------------------------
# Knobs are env-overridable so a full "Run All" isn't punishing:
#   L2_RUN=0 skips it; L2_STEPS / L2_MAX_PATIENTS tune SVI cost.
@dataclass
class Layer2Config:
    run: bool = os.environ.get("L2_RUN", "0") == "1"   # OFF by default — see Layer-2 note
    max_patients: int = int(os.environ.get("L2_MAX_PATIENTS", "1500"))  # SVI subsample cap
    n_folds: int = 5              # patient-level CV
    svi_steps_max: int = int(os.environ.get("L2_STEPS", "1500"))
    svi_lr: float = 1e-2
    svi_patience: int = 200       # early stop
    svi_tol: float = 1.0          # ELBO units
L2 = Layer2Config()
COHORT_SUMMARY_PARQUET = DERIVED_DIR / "aki_cohort_summary.parquet"
GOLD_SAMPLE_STAYS = 800   # stays sampled for the M1 mimic-code gold-standard check

# Canonical MIMIC timestamp format (kills the slow dateutil per-element warning).
MIMIC_TS_FMT = "%Y-%m-%d %H:%M:%S"
def to_dt(s):
    return pd.to_datetime(s, format=MIMIC_TS_FMT, errors="coerce")

print("Config loaded (v3.3-faithful + label/ETL fixes).")
print("  States       :", STATE_LABELS, "| emission mode:", EMISSION_MODE)
print("  Output        :", COHORT_PARQUET.name, "(v2 — stale v1 parquet is not loaded)")
print("  MIMIC_ROOT   :", MIMIC_ROOT, "(exists:", MIMIC_ROOT.exists(), ")")


# ============================================================================
# 2 · LAYER 1 — Multi-state severity HMM  (faithful port of PEMDT v3.3)
# ============================================================================
# Ported from PEMDT_v2_stepbystep_v3.3 cells 8/38/40/42:
#   * _logpdf_gauss, age_bracket
#   * default_state_means / default_state_scales / default_transition
#   * the JAX `forward_filter` (jit + lax.scan) — copied VERBATIM
#   * run_filter_patient wrapper (per-patient z-scoring + mask + weights)
# The only adaptation is N_STATES = 4 (KDIGO) instead of 5 (generic severity).
import jax
import jax.numpy as jnp
import numpy as np
from scipy.stats import norm   # used by the LLR attribution below

# ---- helpers (v3.3 cell 8) -------------------------------------------------
def _logpdf_gauss(x, mu, sigma):
    return -0.5 * jnp.log(2 * jnp.pi) - jnp.log(sigma) - 0.5 * ((x - mu) / sigma) ** 2

def age_bracket(age):
    for k, (lo, hi) in enumerate(AGE_BRACKETS):
        if lo <= age < hi:
            return k
    return len(AGE_BRACKETS) - 1

# ---- population emission geometry (v3.3 cell 38) ---------------------------
def default_state_means():
    """means[k, f] = STATE_PROGRESSION[k] * FEATURE_DIRECTION[f]  (in z-units)."""
    means = np.zeros((N_STATES, len(FEATURES)))
    for fi, f in enumerate(FEATURES):
        for ki, delta in enumerate(STATE_PROGRESSION):
            means[ki, fi] = delta * FEATURE_DIRECTION[f]
    return means

def default_state_scales():
    """Wider emission scale for the more-severe regimes (more heterogeneity)."""
    scales = np.ones((N_STATES, len(FEATURES)))
    scales[2, :] = 1.3   # KDIGO_2 (analogue of v3.3 'Severe')
    scales[3, :] = 1.5   # KDIGO_3 (analogue of v3.3 'Critical')
    return scales

def default_transition(age_idx=1):
    """Tridiagonal: +1 escalation (fixed), -1 recovery (age-bracketed). v3.3 cell 38."""
    recovery_by_age = [0.20, 0.15, 0.10, 0.08]
    escalation = 0.05
    rec = recovery_by_age[age_idx]
    A = np.zeros((N_STATES, N_STATES))
    for k in range(N_STATES):
        if k > 0:            A[k, k - 1] = rec
        if k < N_STATES - 1: A[k, k + 1] = escalation
        A[k, k] = 1.0 - A[k].sum()
    return A

STATE_MEANS_POP = default_state_means()
STATE_SCALES_POP = default_state_scales()

# ---- clinically-anchored emissions (FIX, default) --------------------------
# Per-state Gaussian means/scales in RAW clinical units, anchored to the KDIGO
# bands (see CLINICAL_ANCHORS). This replaces population z-scoring, which
# flattened the signal because the cohort itself is full of AKI.
def clinical_state_means():
    M = np.zeros((N_STATES, len(FEATURES)))
    for fi, f in enumerate(FEATURES): M[:, fi] = CLINICAL_ANCHORS[f][0]
    return M
def clinical_state_scales():
    S = np.ones((N_STATES, len(FEATURES)))
    for fi, f in enumerate(FEATURES): S[:, fi] = CLINICAL_ANCHORS[f][1]
    return S
CLINICAL_MEANS = clinical_state_means()
CLINICAL_SCALES = clinical_state_scales()

def active_means_scales():
    """Return (means, scales) for the configured EMISSION_MODE."""
    return (CLINICAL_MEANS, CLINICAL_SCALES) if EMISSION_MODE == "clinical" \
        else (STATE_MEANS_POP, STATE_SCALES_POP)

# ---- the JAX forward filter (v3.3 cell 40 — VERBATIM) ----------------------
@jax.jit
def forward_filter(z_obs, mask, state_means, state_scales, transition, prior, weights):
    """Canonical HMM forward filter.
    z_obs/mask: (T,F); state_means/scales: (K,F); transition: (K,K); prior: (K,);
    weights: (F,). Returns beliefs (T,K). Off-by-one corrected; outputs clipped."""
    z_clean = jnp.where(mask > 0, z_obs, 0.0)
    # t=0: emission applied directly to prior (no prediction step)
    log_em0 = (mask[0][None, :] * weights[None, :] *
               _logpdf_gauss(z_clean[0][None, :], state_means, state_scales)).sum(axis=1)
    log_post0 = jnp.log(prior + 1e-30) + log_em0
    log_post0 = log_post0 - jnp.max(log_post0)
    post0 = jnp.exp(log_post0); post0 = post0 / (post0.sum() + 1e-30)

    def step(belief, inputs):
        z, m = inputs
        log_pred = jnp.log(belief @ transition + 1e-30)
        log_em = (m[None, :] * weights[None, :] *
                  _logpdf_gauss(z[None, :], state_means, state_scales)).sum(axis=1)
        log_post = log_pred + log_em
        log_post = log_post - jnp.max(log_post)
        post = jnp.exp(log_post); post = post / (post.sum() + 1e-30)
        return post, post

    _, tail = jax.lax.scan(step, post0, (z_clean[1:], mask[1:]))
    beliefs = jnp.concatenate([post0[None, :], tail], axis=0)
    return jnp.clip(beliefs, 0.0, 1.0)

# ---- population z-score parameters from TRAIN (v3.3 "Step 1.3") -------------
def compute_global_stats(train_df, features=FEATURES):
    """Population (mu, sd) per feature, used to z-score emissions. Robust to NaN."""
    stats = {}
    for f in features:
        v = pd.to_numeric(train_df[f], errors="coerce").dropna()
        lo, hi = VALID_RANGES.get(f, (-np.inf, np.inf))
        v = v[(v > lo) & (v < hi)]
        mu = float(v.mean()) if len(v) else GLOBAL_STATS[f][0]
        sd = float(v.std(ddof=0)) if len(v) > 1 and v.std(ddof=0) > 1e-6 else GLOBAL_STATS[f][1]
        stats[f] = (mu, sd)
    return stats

# personal_stats hook (Layer-2 fills this; falls back to global for Layer 1)
personal_stats: dict = {}

# ---- per-patient wrapper (v3.3 cell 42) ------------------------------------
def run_filter_patient(group, state_means=None, state_scales=None, global_stats=None,
                       weights=None, use_personal=True, prior=PRIOR_BELIEF,
                       emission_mode=None, rrt_absorb=True):
    """Forward-filter one stay.

    emission_mode='clinical' -> feed RAW feature values against CLINICAL_MEANS
      (no z-scoring; global_stats unused).
    emission_mode='zscore'   -> per-patient/global z-scores against STATE_MEANS_POP.
    rrt_absorb=True          -> once rrt_on fires, belief is pinned to Stage 3
                                (RRT is an absorbing severe state)."""
    mode = emission_mode or EMISSION_MODE
    if state_means is None or state_scales is None:
        state_means, state_scales = active_means_scales()
    pid = group["stay_id"].iloc[0]
    pat_stats = (personal_stats.get(pid, global_stats) if (use_personal and global_stats) else global_stats)
    age = float(group["anchor_age"].iloc[0]) if "anchor_age" in group.columns and \
        pd.notna(group["anchor_age"].iloc[0]) else 65.0
    trans = default_transition(age_bracket(age))

    obs = np.zeros((len(group), len(FEATURES)))
    msk = np.zeros_like(obs)
    for j, f in enumerate(FEATURES):
        raw = pd.to_numeric(group[f], errors="coerce").values
        lo, hi = VALID_RANGES.get(f, (-np.inf, np.inf))
        ok = ~np.isnan(raw) & (raw > lo) & (raw < hi)
        if mode == "clinical":
            obs[ok, j] = raw[ok]
        else:
            mu, sd = (pat_stats or GLOBAL_STATS)[f]
            obs[ok, j] = (raw[ok] - mu) / sd
        msk[ok, j] = 1.0

    if weights is None:
        weights = np.array([FEATURE_WEIGHTS[f] for f in FEATURES])
    beliefs = np.asarray(forward_filter(
        jnp.array(obs), jnp.array(msk), jnp.array(state_means), jnp.array(state_scales),
        jnp.array(trans), jnp.array(prior), jnp.array(weights)))
    beliefs = np.clip(beliefs, 0.0, 1.0)

    # RRT -> absorbing Stage 3 (creatinine no longer reflects renal function).
    if rrt_absorb and "rrt_on" in group.columns:
        on = pd.to_numeric(group["rrt_on"], errors="coerce").fillna(0).values > 0
        if on.any():
            onehot = np.zeros(N_STATES); onehot[ABSORBING_RRT_STATE] = 1.0
            beliefs[on] = onehot

    out = group.copy().reset_index(drop=True)
    for k, lbl in enumerate(STATE_LABELS):
        out[f"p_{lbl}"] = beliefs[:, k]
    out["p_AKI"] = 1.0 - beliefs[:, 0]
    out["risk_score"] = np.clip(beliefs[:, 2] + beliefs[:, 3], 0.0, 1.0)  # p(KDIGO>=2)
    return out

# ---- glass-box LLR attribution (z-score form, matches v3.3 emission) -------
FEATURE_LABELS = {"creatinine_ratio": "rising creatinine", "dSCr_48h": "48h creatinine jump",
                  "uo_rate": "falling urine output", "bun": "rising BUN"}

def attribute_window(row, global_stats=None, from_state=0, to_state=1, top_k=2,
                     emission_mode=None):
    """LLR per feature between adjacent regimes, in the engine's own space."""
    mode = emission_mode or EMISSION_MODE
    m, s = active_means_scales()
    best = []
    for f in FEATURES:
        x = pd.to_numeric(pd.Series([row.get(f)]), errors="coerce").iloc[0]
        if np.isnan(x):
            continue
        fi = FEATURES.index(f)
        if mode == "clinical":
            val = x
        else:
            mu, sd = (global_stats or GLOBAL_STATS)[f]; val = (x - mu) / sd
        llr = (norm.logpdf(val, m[to_state, fi], s[to_state, fi])
               - norm.logpdf(val, m[from_state, fi], s[from_state, fi])) * FEATURE_WEIGHTS[f]
        best.append((f, float(llr)))
    best.sort(key=lambda kv: kv[1], reverse=True)
    drivers = [FEATURE_LABELS[f] for f, v in best[:top_k] if v > 0]
    sev = STATE_LABELS[to_state]
    return (f"{sev} risk rising: " + " + ".join(drivers)) if drivers else f"{sev} risk stable"

print("Layer-1 engine ported (JAX forward_filter, default_state_means/scales/transition,")
print("run_filter_patient, compute_global_stats, attribute_window).  Devices:", jax.devices())


# ============================================================================
# 3 · KDIGO 2012 STAGING  —  ground-truth labels on the hourly grid
# ============================================================================
# Stage(t) = max(creatinine-stage, urine-output-stage, RRT-stage).
# These produce the *labels* we validate against — the latent regime the engine
# infers is a separate, earlier-moving quantity.

def creatinine_stage(ratio, dscr_48h, scr):
    """KDIGO creatinine arm. ratio=SCr/baseline, dscr_48h=max rise over 48 h."""
    ratio = -np.inf if ratio is None or (isinstance(ratio, float) and np.isnan(ratio)) else ratio
    dscr_48h = -np.inf if dscr_48h is None or (isinstance(dscr_48h, float) and np.isnan(dscr_48h)) else dscr_48h
    scr = -np.inf if scr is None or (isinstance(scr, float) and np.isnan(scr)) else scr
    if ratio >= KDIGO.ratio_stage3 or scr >= KDIGO.scr_stage3_abs:
        return 3
    if ratio >= KDIGO.ratio_stage2:
        return 2
    if ratio >= KDIGO.ratio_stage1 or dscr_48h >= KDIGO.abs_rise_48h:
        return 1
    return 0


def urine_output_stage(uo_6h, uo_12h, uo_24h):
    """KDIGO urine-output arm from rolling mean rates (mL/kg/h) over windows.

    Hourly-grid approximation of the 'sustained for >= N h' criteria: a window
    mean below threshold means the rate stayed low across that window. Anuria is
    treated as ~0 mL/kg/h sustained >= 12 h (captured by the 12 h window ~ 0)."""
    def low(x, thr):
        return (x is not None) and (not (isinstance(x, float) and np.isnan(x))) and (x < thr)
    # Stage 3: < 0.3 for >= 24 h, OR anuria (~0) for >= 12 h.
    if low(uo_24h, KDIGO.uo_s3_rate) or (uo_12h is not None and not np.isnan(uo_12h) and uo_12h < 0.05):
        return 3
    if low(uo_12h, KDIGO.uo_low_rate):     # < 0.5 for >= 12 h
        return 2
    if low(uo_6h, KDIGO.uo_low_rate):      # < 0.5 for 6-12 h
        return 1
    return 0


def kdigo_stage_row(row):
    """max() of the three arms for one harmonized hour-row."""
    c = creatinine_stage(row.get("creatinine_ratio"), row.get("dSCr_48h"), row.get("creatinine"))
    u = urine_output_stage(row.get("uo_6h"), row.get("uo_12h"), row.get("uo_24h"))
    r = 3 if row.get("rrt_on", 0) else 0
    return int(max(c, u, r))


def add_kdigo_stage(df):
    """Vectorized-ish stage derivation for a harmonized per-stay-hour frame."""
    df = df.copy()
    df["kdigo_cr"]  = [creatinine_stage(a, b, c) for a, b, c in
                       zip(df.get("creatinine_ratio"), df.get("dSCr_48h"), df.get("creatinine"))]
    df["kdigo_uo"]  = [urine_output_stage(a, b, c) for a, b, c in
                       zip(df.get("uo_6h"), df.get("uo_12h"), df.get("uo_24h"))]
    df["kdigo_rrt"] = np.where(df.get("rrt_on", 0) > 0, 3, 0)
    df["kdigo_stage"] = df[["kdigo_cr", "kdigo_uo", "kdigo_rrt"]].max(axis=1).astype(int)
    return df


def rolling_uo_rates(stay_df, uo_rate_col="uo_rate"):
    """Add uo_6h/uo_12h/uo_24h rolling-mean rates (mL/kg/h) on an hourly grid.

    Assumes stay_df is one stay, sorted by `hour`, with `uo_rate` already in
    mL/kg/h per hour (0 where charted-and-empty, NaN where charting absent)."""
    s = stay_df.sort_values("hour").copy()
    r = s[uo_rate_col]
    s["uo_6h"]  = r.rolling(KDIGO.uo_s1_hours, min_periods=KDIGO.uo_s1_hours).mean()
    s["uo_12h"] = r.rolling(KDIGO.uo_s2_hours, min_periods=KDIGO.uo_s2_hours).mean()
    s["uo_24h"] = r.rolling(KDIGO.uo_s3_hours, min_periods=KDIGO.uo_s3_hours).mean()
    return s

print("KDIGO staging loaded: creatinine_stage, urine_output_stage, add_kdigo_stage, rolling_uo_rates.")


# ============================================================================
# 4 · PHASE 0 ETL  —  cohort, baseline, hourly harmonization -> aki_cohort.parquet
# ============================================================================
# The "unglamorous 40%". chartevents (~40 GB) and labevents (~18 GB) cannot be
# loaded whole, so we pre-filter to the item IDs above with DuckDB streaming
# (falls back to chunked pandas). Each helper degrades gracefully and tells you
# exactly which file is missing rather than dying obscurely.
import glob as _glob

try:
    import duckdb
    _HAS_DUCKDB = True
except Exception:
    _HAS_DUCKDB = False


if not _HAS_DUCKDB:
    print("NOTE: duckdb not importable in this kernel -> using slower chunked-pandas "
          "streaming. `pip install duckdb` in THIS kernel for a big speedup.")

def _ensure_dt(df, col="charttime"):
    """Return a copy of df with `col` guaranteed datetime64 (merge_asof needs
    both sides the same dtype). No-op if already datetime."""
    if df is None or col not in getattr(df, "columns", []):
        return df
    if not pd.api.types.is_datetime64_any_dtype(df[col]):
        df = df.copy(); df[col] = to_dt(df[col])
    return df


def _resolve(path_noext):
    """Return the existing .csv or .csv.gz path for a MIMIC table stem, or None."""
    for ext in (".csv", ".csv.gz"):
        p = Path(str(path_noext) + ext)
        if p.exists():
            return p
    hits = _glob.glob(str(path_noext) + "*")
    return Path(hits[0]) if hits else None


def _file_columns(path):
    """Column names of a (possibly gzipped) CSV without reading the body."""
    if _HAS_DUCKDB:
        return list(duckdb.query(f"SELECT * FROM read_csv_auto('{path}') LIMIT 0").df().columns)
    return list(pd.read_csv(path, nrows=0).columns)


# MIMIC-IV is INCONSISTENT about the numeric value column: labevents/chartevents
# use `valuenum`; outputevents/procedureevents use `value`; inputevents uses
# `amount`. We auto-detect whichever exists and alias it to the requested name,
# so callers (and harmonize) can always rely on a single column name. (FIX for
# the `KeyError: 'valuenum'` — the old pandas fallback silently dropped a value
# column that the file didn't have, then code asked for it.)
_VALUE_ALIASES = ["valuenum", "value", "amount"]

def stream_items(table_stem, itemids, value_col="valuenum", id_col="stay_id",
                 time_col="charttime", subject_ids=None, extra_cols=()):
    """Pre-filter a big MIMIC events table to the given item IDs. Returns a tidy
    DataFrame with the numeric value under the name `value_col`, regardless of
    whether the file calls it value/valuenum/amount. DuckDB if available, else
    chunked pandas."""
    path = _resolve(table_stem)
    if path is None:
        raise FileNotFoundError(
            f"Missing MIMIC table '{table_stem}.csv[.gz]'. Check MIMIC_ROOT={MIMIC_ROOT}.")
    avail = set(_file_columns(path))
    # Resolve the real value column to read (then alias -> value_col).
    src = value_col if value_col in avail else next((c for c in _VALUE_ALIASES if c in avail), None)
    keep = [c for c in [id_col, time_col, "itemid", *extra_cols] if c in avail]
    idlist = ",".join(str(i) for i in itemids)

    if _HAS_DUCKDB:
        valexpr = f"{src} AS {value_col}" if src else f"CAST(NULL AS DOUBLE) AS {value_col}"
        sel = ", ".join(keep + [valexpr])
        q = (f"SELECT {sel} FROM read_csv_auto('{path}', union_by_name=true) "
             f"WHERE itemid IN ({idlist})")
        if subject_ids is not None and "subject_id" in avail:
            q += " AND subject_id IN (" + ",".join(map(str, subject_ids)) + ")"
        return duckdb.query(q).to_df()

    # pandas fallback
    usecols = set(keep) | ({src} if src else set())
    out = []
    for chunk in pd.read_csv(path, usecols=lambda c: c in usecols, chunksize=2_000_000):
        sub = chunk[chunk["itemid"].isin(itemids)]
        if subject_ids is not None and "subject_id" in sub.columns:
            sub = sub[sub["subject_id"].isin(subject_ids)]
        if len(sub):
            out.append(sub)
    res = pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=list(usecols))
    if src and src != value_col:
        res = res.rename(columns={src: value_col})
    elif not src:
        res[value_col] = np.nan
    return res


# ---- Step 0.1 / 0.2 : cohort + exclusions ----------------------------------
def build_cohort_index():
    """Adult ICU stays, LOS >= 1.5 d, >= 2 creatinine measurements; exclude
    pre-existing ESRD / chronic dialysis / transplant / RRT-at-admission.
    Returns a per-stay DataFrame (one row per stay_id) with flags."""
    icustays  = pd.read_csv(_resolve(ICU_DIR / "icustays"))
    patients  = pd.read_csv(_resolve(HOSP_DIR / "patients"))
    admissions = pd.read_csv(_resolve(HOSP_DIR / "admissions"))
    dx_path = _resolve(HOSP_DIR / "diagnoses_icd")
    diagnoses = pd.read_csv(dx_path) if dx_path else pd.DataFrame(columns=["subject_id", "icd_code"])

    df = icustays.merge(patients[["subject_id", "anchor_age", "gender", "dod"]], on="subject_id", how="left")
    df = df[df["anchor_age"] >= COHORT.min_age]
    df = df[df["los"] >= COHORT.min_los_days]

    # ESRD / chronic dialysis / transplant exclusion via ICD-9/10.
    esrd_codes = {"5856", "N186", "Z992", "V4511", "Z9911"}        # ESRD + dialysis status
    txp_codes  = {"V420", "Z940", "T861", "9960", "0TY00Z0"}        # kidney transplant
    bad = esrd_codes | txp_codes
    diagnoses["icd_norm"] = diagnoses["icd_code"].astype(str).str.replace(".", "", regex=False).str.upper()
    excl_subjects = set(diagnoses.loc[diagnoses["icd_norm"].isin(bad), "subject_id"])
    df["excl_ckd_esrd_txp"] = df["subject_id"].isin(excl_subjects)

    # >= 2 creatinine measurements during the admission.
    scr = stream_items(HOSP_DIR / "labevents", CREATININE_ITEMIDS,
                       value_col="valuenum", id_col="hadm_id", time_col="charttime",
                       extra_cols=("subject_id",))
    scr_counts = scr.groupby("subject_id").size()
    df["n_creatinine"] = df["subject_id"].map(scr_counts).fillna(0).astype(int)
    df = df[df["n_creatinine"] >= COHORT.min_creatinine_measurements]

    df = df[~df["excl_ckd_esrd_txp"]].copy()
    print(f"Cohort: {df['stay_id'].nunique()} stays / {df['subject_id'].nunique()} subjects after exclusions.")
    return df.reset_index(drop=True), scr


# ---- Step 0.3 : baseline creatinine ----------------------------------------
def baseline_creatinine(scr_subject, intime, gender, age):
    """Primary: lowest SCr in the 7 d before ICU admission. Fallback: MDRD
    back-estimate assuming eGFR=75. Returns (baseline_mgdl, method)."""
    pre = scr_subject[(scr_subject["charttime"] < intime) &
                      (scr_subject["charttime"] >= intime - pd.Timedelta(days=7))]
    if len(pre):
        return float(pre["valuenum"].min()), "7d_min"
    # MDRD solved for SCr at eGFR=75: SCr = (eGFR / (175 * age^-0.203 * sex * race))^(1/-1.154)
    sex = 0.742 if str(gender).upper().startswith("F") else 1.0
    k = 175.0 * (age ** -0.203) * sex                # race factor omitted (not in MIMIC)
    scr_est = (COHORT.default_egfr / k) ** (1.0 / -1.154)
    return float(scr_est), "mdrd75"


# ---- Step 0.4 : weight ------------------------------------------------------
def _valid_weight(x):
    lo, hi = WEIGHT_VALID
    return x if (x is not None and lo <= x <= hi) else None

def resolve_weight(stay_id, subject_id, weights_df, inputs_weight=None, cohort_median=80.0):
    """admit wt -> daily wt -> inputevents.patientweight -> cohort median.
    Each candidate is range-checked (20-300 kg) so a bad weight can't blow up
    the mL/kg/h rate (FIX: previously a near-zero weight gave uo_rate ~ 1e5)."""
    w = weights_df[weights_df["stay_id"] == stay_id]
    admit = w[w["itemid"] == 226512]["valuenum"].dropna()
    if len(admit):
        v = _valid_weight(float(admit.iloc[0]))
        if v: return v
    daily = w[w["itemid"] == 224639]["valuenum"].dropna()
    if len(daily):
        v = _valid_weight(float(daily.median()))
        if v: return v
    if inputs_weight is not None and len(inputs_weight):
        v = _valid_weight(float(inputs_weight.median()))
        if v: return v
    return float(cohort_median)


# ---- Step 0.5 / 0.6 : harmonize one stay to the hourly grid -----------------
def harmonize_patient(stay_row, scr_subject, uo_stay, weight, bun_subject=None,
                      map_stay=None, vaso_stay=None, nephrotox_events=None, rrt_times=None):
    """Build the per-stay-hour o_t table (the AKI analogue of CGM harmonize).

    Columns: creatinine (LOCF), creatinine_ratio, dSCr_48h, uo_rate, uo_6h/12h/24h,
    bun, MAP, vaso_rate, nephrotoxin_on_board, sepsis_flag(placeholder), rrt_on,
    plus kdigo_stage. One row per stay-hour."""
    intime  = to_dt(stay_row["intime"])
    outtime = to_dt(stay_row["outtime"])
    hours = pd.date_range(intime.floor("h"), outtime.ceil("h"), freq=COHORT.grid_freq)
    g = pd.DataFrame({"charttime": hours})
    g["stay_id"] = stay_row["stay_id"]; g["subject_id"] = stay_row["subject_id"]
    g["anchor_age"] = stay_row.get("anchor_age", 65)   # for age-bracketed transitions (v3.3)
    g["hour"] = np.arange(len(g))

    # Guarantee datetime keys on every right-hand frame before any merge_asof
    # (MIMIC streams arrive as object strings unless pre-converted -> dtype clash).
    scr_subject = _ensure_dt(scr_subject); bun_subject = _ensure_dt(bun_subject)
    map_stay = _ensure_dt(map_stay)

    base, base_method = baseline_creatinine(scr_subject, intime, stay_row.get("gender"), stay_row.get("anchor_age", 60))
    g["baseline_creatinine"] = base; g["baseline_method"] = base_method

    # Creatinine: align draws to the grid, LOCF between draws.
    scr_in = scr_subject[(scr_subject["charttime"] >= intime) & (scr_subject["charttime"] <= outtime)]
    scr_in = scr_in.sort_values("charttime")
    g = pd.merge_asof(g, scr_in[["charttime", "valuenum"]].rename(columns={"valuenum": "creatinine"}),
                      on="charttime", direction="backward")
    g["creatinine"] = g["creatinine"].ffill()
    g["creatinine_ratio"] = g["creatinine"] / base
    # 48 h rise computed on the ACTUAL DRAWS, then LOCF onto the grid (FIX:
    # the old hourly rolling-min over LOCF values cancelled to ~0 everywhere).
    if len(scr_in):
        d = scr_in[["charttime", "valuenum"]].copy()
        roll_min = d.set_index("charttime")["valuenum"].rolling("48h", min_periods=1).min()
        d["dSCr_48h"] = d["valuenum"].values - roll_min.values
        g = pd.merge_asof(g, d[["charttime", "dSCr_48h"]], on="charttime", direction="backward")
        g["dSCr_48h"] = g["dSCr_48h"].ffill().fillna(0.0)
    else:
        g["dSCr_48h"] = np.nan

    # Urine output -> hourly NET mL (irrigant-in subtracted), then mL/kg/h rate.
    if uo_stay is not None and len(uo_stay):
        uo = _ensure_dt(uo_stay.copy())
        v = pd.to_numeric(uo["valuenum"], errors="coerce")
        # GU Irrigant Volume In (227488) is fluid going IN -> negate (FIX).
        sign = np.where(uo["itemid"].isin(URINE_NEGATIVE_ITEMIDS), -1.0, 1.0)
        uo["signed"] = v.values * sign
        uo_hourly = uo.set_index("charttime")["signed"].resample(COHORT.grid_freq).sum()
        g = g.merge(uo_hourly.rename("uo_ml"), on="charttime", how="left")
        # only HOURS WITH CHARTING are scored; empty hours stay NaN (not 0).
        charted = uo.set_index("charttime")["signed"].resample(COHORT.grid_freq).count()
        g = g.merge(charted.rename("_uo_n"), on="charttime", how="left")
        g["uo_charted"] = g["_uo_n"].fillna(0) > 0
        g.loc[~g["uo_charted"], "uo_ml"] = np.nan
        g["uo_ml"] = g["uo_ml"].clip(lower=0)                # net flush can go negative -> 0
        g["uo_rate"] = (g["uo_ml"] / max(weight, 1e-3)).clip(*UO_RATE_CLIP)  # clip artifacts
        g = g.drop(columns=["_uo_n"])
    else:
        g["uo_ml"] = np.nan; g["uo_charted"] = False; g["uo_rate"] = np.nan
    g = rolling_uo_rates(g)                                   # uo_6h/12h/24h

    # BUN (context).
    g["bun"] = np.nan
    if bun_subject is not None and len(bun_subject):
        b = bun_subject.sort_values("charttime")
        g = pd.merge_asof(g.drop(columns=["bun"]), b[["charttime", "valuenum"]].rename(columns={"valuenum": "bun"}),
                          on="charttime", direction="backward")
        g["bun"] = g["bun"].ffill()

    # MAP / vaso (context + Phase-5 hypotension input).
    g["MAP"] = np.nan
    if map_stay is not None and len(map_stay):
        m = map_stay.sort_values("charttime")
        g = pd.merge_asof(g.drop(columns=["MAP"]), m[["charttime", "valuenum"]].rename(columns={"valuenum": "MAP"}),
                          on="charttime", direction="backward")
    g["vaso_rate"] = 0.0
    if vaso_stay is not None and len(vaso_stay):
        # crude: mark hours overlapping any vaso administration window.
        for _, ev in vaso_stay.iterrows():
            mask = (g["charttime"] >= to_dt(ev["starttime"])) & (g["charttime"] <= to_dt(ev["endtime"]))
            g.loc[mask, "vaso_rate"] = float(ev.get("rate", 1.0) or 1.0)

    # Nephrotoxin-on-board: exponential decay since each exposure (the Phase-5 signal).
    g["nephrotoxin_on_board"] = 0.0
    if nephrotox_events is not None and len(nephrotox_events):
        lam = np.log(2) / COHORT.nephrotox_halflife_h
        for _, ev in nephrotox_events.iterrows():
            t0 = to_dt(ev["starttime"])
            dt_h = (g["charttime"] - t0).dt.total_seconds() / 3600.0
            contrib = np.where(dt_h >= 0, np.exp(-lam * dt_h), 0.0)
            g["nephrotoxin_on_board"] += contrib

    g["sepsis_flag"] = 0          # placeholder: join sepsis-3 concept if desired
    g["cumulative_fluid_balance"] = np.nan

    # RRT -> absorbing Stage 3 from first event.
    g["rrt_on"] = 0
    if rrt_times is not None and len(rrt_times):
        rt = [to_dt(t) for t in rrt_times]
        rt = [t for t in rt if pd.notna(t)]
        if rt:
            g.loc[g["charttime"] >= min(rt), "rrt_on"] = 1

    g = add_kdigo_stage(g)
    return g


def build_cohort(limit=None, save=True):
    """Full Phase-0 orchestrator. Returns aki_cohort (per-stay-hour) and writes
    aki_cohort.parquet. Streams events once and caches per-stay frames."""
    cohort, scr_all = build_cohort_index()
    scr_all["charttime"] = to_dt(scr_all["charttime"])
    if limit:
        cohort = cohort.head(limit)
    subj = set(cohort["subject_id"])

    bun_all = stream_items(HOSP_DIR / "labevents", BUN_ITEMIDS, id_col="hadm_id", extra_cols=("subject_id",))
    bun_all["charttime"] = to_dt(bun_all["charttime"])
    uo_all  = stream_items(ICU_DIR / "outputevents", UO_ITEMIDS, extra_cols=("subject_id",))
    wt_all  = stream_items(ICU_DIR / "chartevents", WEIGHT_ITEMIDS, extra_cols=("subject_id",))
    map_all = stream_items(ICU_DIR / "chartevents", MAP_ITEMIDS, extra_cols=("subject_id",))
    nephro_iv = stream_items(ICU_DIR / "inputevents", list(NEPHROTOX_IV_ITEMIDS),
                             value_col="amount", time_col="starttime", extra_cols=("subject_id", "endtime"))
    # RRT from procedureevents (FIX: was never streamed -> rrt_on was always 0).
    rrt_all = stream_items(ICU_DIR / "procedureevents", RRT_PROCEDURE_ITEMIDS,
                           value_col="value", time_col="starttime", extra_cols=("subject_id",))
    cohort_median_wt = 80.0

    frames = []
    for _, stay in cohort.iterrows():
        sid, subject = stay["stay_id"], stay["subject_id"]
        rrt_stay = rrt_all[rrt_all["stay_id"] == sid] if ("stay_id" in rrt_all and len(rrt_all)) else None
        g = harmonize_patient(
            stay,
            scr_subject=scr_all[scr_all["subject_id"] == subject],
            uo_stay=uo_all[uo_all["stay_id"] == sid] if "stay_id" in uo_all else None,
            weight=resolve_weight(sid, subject, wt_all, cohort_median=cohort_median_wt),
            bun_subject=bun_all[bun_all["subject_id"] == subject],
            map_stay=map_all[map_all["stay_id"] == sid] if "stay_id" in map_all else None,
            nephrotox_events=nephro_iv[nephro_iv["stay_id"] == sid] if "stay_id" in nephro_iv else None,
            rrt_times=(rrt_stay["starttime"].tolist() if rrt_stay is not None and len(rrt_stay) else None),
        )
        frames.append(g)
    cohort_hourly = pd.concat(frames, ignore_index=True)
    if save:
        cohort_hourly.to_parquet(COHORT_PARQUET, index=False)
        print(f"Wrote {COHORT_PARQUET}  ({len(cohort_hourly):,} stay-hours).")
    return cohort_hourly

print("Phase-0 ETL loaded: stream_items, build_cohort_index, harmonize_patient, build_cohort.")
print("Run build_cohort(limit=50) once MIMIC_ROOT is mounted to produce aki_cohort.parquet.")



Config loaded (v3.3-faithful + label/ETL fixes).
  States       : ['No_AKI', 'KDIGO_1', 'KDIGO_2', 'KDIGO_3'] | emission mode: clinical
  Output        : aki_cohort_v2.parquet (v2 — stale v1 parquet is not loaded)
  MIMIC_ROOT   : /Users/sumiyasayeed/Documents/mimic4/mimiciv (exists: True )
Layer-1 engine ported (JAX forward_filter, default_state_means/scales/transition,
run_filter_patient, compute_global_stats, attribute_window).  Devices: [CpuDevice(id=0)]
KDIGO staging loaded: creatinine_stage, urine_output_stage, add_kdigo_stage, rolling_uo_rates.
NOTE: duckdb not importable in this kernel -> using slower chunked-pandas streaming. `pip install duckdb` in THIS kernel for a big speedup.
Phase-0 ETL loaded: stream_items, build_cohort_index, harmonize_patient, build_cohort.
Run build_cohort(limit=50) once MIMIC_ROOT is mounted to produce aki_cohort.parquet.


## 2 · Relabel UO arm with mimic-code method → new parquet


In [2]:
# ============================================================================
# EXPERIMENT A — relabel the UO arm with mimic-code's uo_tm method
# ============================================================================
# NON-DESTRUCTIVE: reads aki_cohort_v2.parquet, recomputes the UO arm on a
# sample using mimic-code's total-urine / uo_tm rule, writes a NEW parquet
# (aki_cohort_v2_mcuo.parquet). The original parquet + main notebook are untouched.
import time, os
EXP_STAYS = int(os.environ.get("EXP_STAYS", "10000"))   # sample size; set 0 for the FULL cohort
t0 = time.time()

# mimic-code UO stage per urine charttime (identical to notebook cell 6d, unit-tested).
def _uo_mc_one(g, weight):
    g = g.sort_values("charttime").reset_index(drop=True); w = max(weight, 1e-3)
    g["hrs_prev"] = g["charttime"].diff().dt.total_seconds().div(3600); g.loc[0, "hrs_prev"] = 1.0
    gi = g.set_index("charttime")
    for h, win in [("6", "6h"), ("12", "12h"), ("24", "24h")]:
        g["uo" + h] = gi["uo"].rolling(win, closed="both").sum().values
        g["tm" + h] = gi["hrs_prev"].rolling(win, closed="both").sum().values
    g["rt6"]  = np.where((g.tm6 >= 6) & (g.tm6 < 12), g.uo6 / w / g.tm6, np.nan)
    g["rt12"] = np.where(g.tm12 >= 12, g.uo12 / w / g.tm12, np.nan)
    g["rt24"] = np.where(g.tm24 >= 24, g.uo24 / w / g.tm24, np.nan)
    sec = (g["charttime"] - g["intime"]).dt.total_seconds().values
    st = np.zeros(len(g), int)
    for i in range(len(g)):
        if sec[i] <= 6 * 3600:
            continue
        r = g.iloc[i]
        if (r.tm24 >= 24 and r.rt24 < 0.3) or (r.tm12 >= 12 and r.rt12 == 0):
            st[i] = 3
        elif r.tm12 >= 12 and r.rt12 < 0.5:
            st[i] = 2
        elif r.tm6 >= 6 and r.rt6 < 0.5:
            st[i] = 1
    g["gold_uo"] = st
    return g[["stay_id", "charttime", "gold_uo"]]

ch = pd.read_parquet(COHORT_PARQUET); ch["charttime"] = pd.to_datetime(ch["charttime"])
allstays = ch["stay_id"].unique()
samp = (allstays if EXP_STAYS in (0, None)
        else pd.Series(allstays).sample(min(EXP_STAYS, len(allstays)), random_state=7).values)
ch = ch[ch["stay_id"].isin(samp)].sort_values(["stay_id", "hour"]).copy()
print(f"Experiment on {len(samp):,} stays / {len(ch):,} hours. Re-streaming urine...")

icu = pd.read_csv(_resolve(ICU_DIR / "icustays")); icu["intime"] = to_dt(icu["intime"])
icu = icu[icu["stay_id"].isin(samp)][["stay_id", "subject_id", "intime"]]
subs = icu["subject_id"].unique().tolist()
uo = stream_items(ICU_DIR / "outputevents", UO_ITEMIDS, extra_cols=("subject_id",), subject_ids=subs)
uo = uo[uo["stay_id"].isin(samp)].copy(); uo["charttime"] = to_dt(uo["charttime"])
uo["signed"] = pd.to_numeric(uo["valuenum"], errors="coerce").values * np.where(
    uo["itemid"].isin(URINE_NEGATIVE_ITEMIDS), -1.0, 1.0)
net = uo.groupby(["stay_id", "charttime"], as_index=False)["signed"].sum().rename(columns={"signed": "uo"})
net["uo"] = net["uo"].clip(lower=0); net = net.merge(icu, on="stay_id")
wt = stream_items(ICU_DIR / "chartevents", WEIGHT_ITEMIDS, extra_cols=("subject_id",), subject_ids=subs)
wv = pd.to_numeric(wt["valuenum"], errors="coerce"); wmed = float(wv[(wv >= 20) & (wv <= 300)].median() or 80.0)
print(f"  urine rows {len(net):,}; computing mimic-code UO stage ({time.time()-t0:.0f}s)...")

gold = pd.concat([_uo_mc_one(g, resolve_weight(sid, None, wt, cohort_median=wmed))
                  for sid, g in net.groupby("stay_id")], ignore_index=True)

# carry the per-draw UO stage forward onto the hourly grid
parts = []
for sid, g in ch.groupby("stay_id"):
    gd = gold[gold["stay_id"] == sid].sort_values("charttime")
    if len(gd) == 0:
        g["kdigo_uo_mc"] = 0
    else:
        m = pd.merge_asof(g.sort_values("charttime"),
                          gd[["charttime", "gold_uo"]].rename(columns={"gold_uo": "kdigo_uo_mc"}),
                          on="charttime", direction="backward")
        g["kdigo_uo_mc"] = m["kdigo_uo_mc"].ffill().fillna(0).astype(int).values
    parts.append(g)
ch = pd.concat(parts, ignore_index=True)
ch["kdigo_stage_mc"] = ch[["kdigo_cr", "kdigo_uo_mc", "kdigo_rrt"]].max(axis=1).astype(int)
ch.to_parquet(DERIVED_DIR / "aki_cohort_v2_mcuo.parquet", index=False)
print(f"Wrote aki_cohort_v2_mcuo.parquet in {time.time()-t0:.0f}s "
      f"(severe-prevalence old {ch.groupby('stay_id').kdigo_stage.max().ge(2).mean():.1%} "
      f"-> mimic-code-UO {ch.groupby('stay_id').kdigo_stage_mc.max().ge(2).mean():.1%}).")



Experiment on 10,000 stays / 1,252,133 hours. Re-streaming urine...
  urine rows 644,585; computing mimic-code UO stage (122s)...
Wrote aki_cohort_v2_mcuo.parquet in 163s (severe-prevalence old 28.2% -> mimic-code-UO 65.3%).


## 3 · Compare: current labels vs mimic-code-UO labels


In [3]:
# ============================================================================
# EXPERIMENT A — compare: same model, old UO labels vs mimic-code UO labels
# ============================================================================
# The engine's risk_score comes from FEATURES, not labels, so we run it ONCE and
# evaluate it against both label definitions. The escalation target (Stage>=2)
# differs between them because the UO arm changed.
from sklearn.metrics import roc_auc_score, average_precision_score, cohen_kappa_score

exp = pd.read_parquet(DERIVED_DIR / "aki_cohort_v2_mcuo.parquet").sort_values(["stay_id", "hour"])
# same UO emission smoothing as the main pipeline (feature only)
exp["uo_rate"] = exp.groupby("stay_id")["uo_rate"].transform(
    lambda s: s.rolling(UO_EMIT_SMOOTH_H, min_periods=1).mean())
scored = pd.concat([run_filter_patient(g.sort_values("hour")) for _, g in exp.groupby("stay_id")],
                   ignore_index=True)

def fast_labels(df, label_col, H=24, tgt=2):
    rows = []
    for sid, g in df.groupby("stay_id"):
        g = g.sort_values("hour"); st = (g[label_col].to_numpy() >= tgt).astype(int); n = len(st)
        cs = np.concatenate(([0], np.cumsum(st)))
        fut = (cs[np.minimum(n, np.arange(n) + 1 + H)] - cs[np.arange(1, n + 1)] > 0).astype(int)
        g = g.assign(y=fut); rows.append(g[g[label_col] < tgt])
    return pd.concat(rows, ignore_index=True)

print("UO-arm agreement old vs mimic-code (per hour): Cohen's kappa "
      f"{cohen_kappa_score(scored['kdigo_uo'], scored['kdigo_uo_mc']):.3f}\n")
print(f"{'horizon':>8}  {'labels':<14} {'AUROC':>6} {'AUPRC':>6}  {'pos-rate':>8}  {'n':>10}")
out = []
for H in (12, 24, 48):
    for name, col in [("current", "kdigo_stage"), ("mimic-code UO", "kdigo_stage_mc")]:
        lab = fast_labels(scored, col, H)
        if lab["y"].nunique() < 2:
            continue
        au = roc_auc_score(lab["y"], lab["risk_score"]); ap = average_precision_score(lab["y"], lab["risk_score"])
        print(f"{H:>6}h  {name:<14} {au:6.3f} {ap:6.3f}  {lab['y'].mean():8.3f}  {len(lab):>10,}")
        out.append({"horizon_h": H, "labels": name, "AUROC": round(au, 4),
                    "AUPRC": round(ap, 4), "pos_rate": round(lab["y"].mean(), 4), "n": len(lab)})
pd.DataFrame(out).to_csv(DERIVED_DIR / "uo_experiment_comparison.csv", index=False)
sev_old = exp.groupby("stay_id").kdigo_stage.max().ge(2).mean()
sev_new = exp.groupby("stay_id").kdigo_stage_mc.max().ge(2).mean()
print(f"\nSevere-AKI prevalence: current {sev_old:.1%}  ->  mimic-code UO {sev_new:.1%}")
print("Saved uo_experiment_comparison.csv. Decision: if mimic-code-UO AUROC is close to current,")
print("the more-defensible labels cost little — adopt; if it drops a lot, keep current + report.")



UO-arm agreement old vs mimic-code (per hour): Cohen's kappa 0.147

 horizon  labels          AUROC  AUPRC  pos-rate           n
    12h  current         0.816  0.174     0.038   1,061,405
    12h  mimic-code UO   0.709  0.285     0.144     891,126
    24h  current         0.779  0.226     0.063   1,061,405
    24h  mimic-code UO   0.653  0.373     0.241     891,126
    48h  current         0.745  0.258     0.090   1,061,405
    48h  mimic-code UO   0.627  0.459     0.338     891,126

Severe-AKI prevalence: current 28.2%  ->  mimic-code UO 65.3%
Saved uo_experiment_comparison.csv. Decision: if mimic-code-UO AUROC is close to current,
the more-defensible labels cost little — adopt; if it drops a lot, keep current + report.
